### Ein Versuch, die verschiedenen Suchstrategien zu illustrieren
- Alle Suchstrategien führen ein Liste `nodes_to_visit` mit Knoten, die
noch besucht werden sollen.
Die Knoten in `nodes_to_visit` bilden die **Front** des erkundeten Teils des Suchraums.  
**Die Suchstrategie gibt vor, welcher Knoten als nächstes besucht wird**. 
- Wird ein Knoten in `nodes_to_visit` besucht, wird dieser aus der Liste entfernt.
  Alle benachbarten noch unbesuchten Nachbarn, die noch nicht in der Liste sind, werden
  hinzugefügt.
- Bei manchen Suchstrategien werden die Knoten in `nodes_to_visit` mit einer Priority versehen. 
  Der Knoten mit der kleinsten Priority wird als nächstes besucht.

1. Depth-first Search: Neue Knoten werden **von rechts** in die Liste **eingefügt** und **von rechts entfernt**.
   Die Suche wird, falls möglich, vom letzten neuen Knoten fortgesetzt.  
   - Zusammenhangskomponenten: Finden von Gruppen von Knoten, in denen jeder Knoten von jedem anderen aus erreichbar ist.
   - Damenproblem (Teillösung falls möglich erweitern)
   - Sudoku-Solver (Teillösung falls möglich erweitern)<br><br>
     
1. Bread-first Search: Neue Knoten werden **von links** in die Liste **eingefügt** und **von rechts entfernt**.
   Neu hinzugefügte Knoten werden erst besucht, nachdem alle andern Knoten in der Liste/Deque besucht wurden.  
   Zuerst werden Knoten mit Abstand 1 von Start besucht, dann Knoten mit Abstand 2, ...
   - Kürzeste Wege finden.
   - Abstand aller Knoten von einen gegebenen Startknoten ermitteln.<br><br>
  
1. Greedy Search: Jedem Knoten wird der **geschätzte Abstand zum Ziel als Priority** zugeordent.
   Der Knoten mit der kleinsten Priority wird als nächstes besucht.
   - Ein (nicht notwendigerweise kürzester) Pfad zum Ziel finden.<br><br>

1. Smart bread-first Search:
   Jedem Knoten wird die **länge des aktuellen Pfads plus der geschätzte Abstand zum Ziel** als Priority   zugeordent. Der Knoten mit der kleinsten Priority wird als nächstes besucht. Die **Priority ist die geschätzte 
Länge des Pfads**.
   - **Die gesamte Suchfront breitet sich in Richtung Ziel aus**.
   - Wird der Weg ins Ziel konservativ geschätzt, wird oft ein kürzester Weg gefunden.<br><br>


1. Bidirectionale bread-first Search: Breath first Search von Start und Ziel aus starten.  
   Hat jeder Knoten im Durschnitt 2 oder mehr neue Nachbarn, reduziert sich die Zahl der zu besuchenden Knoten stark.
   - Lösung beim 2x2x2 Rubiks-Cube suchen.

1. Smart bidirectional Search: Smart breath-first Search von Start und Ziel aus starten.

In [ ]:
import grid_controller as C
import game
import view


block_mode = False
clicked = None
is_mouse_down = False


def set_clicked(val=None):
    global clicked
    clicked = val


def toggle_knightmoves():
    game.state['knightmoves'] = not game.state['knightmoves']
    print(f'knight moves: {game.state['knightmoves']}')


def downsize():
    ncol, nrow = game.state['dims']
    if ncol <= 4:
        return
    game.state['dims'] = (ncol - 2, nrow - 2)
    game.reset()


def upsize():
    ncol, nrow = game.state['dims']
    if ncol >= 30:
        return
    game.state['dims'] = (ncol + 2, nrow + 2)
    game.reset()


def toggle_blockmode():
    global block_mode
    block_mode = not block_mode
    set_clicked()
    view.draw_state()
    print(f'block mode: {block_mode}')


def on_mouse_down(pos):
    global clicked, is_mouse_down
    is_mouse_down = True
    if block_mode:
        pass
    elif pos == clicked:
        clicked = None
        view.draw_state()
    elif clicked is None:
        clicked = pos
        view.G.fill_rect(view.canvas, pos, view.grid_spec, color='green')
    else:
        game.change_state(('start', 'goal'), (clicked, pos))
        clicked = None
        view.draw_state()


def on_mouse_up(pos):
    global is_mouse_down
    is_mouse_down = False


def on_mouse_move(pos):
    if not (is_mouse_down and block_mode):
        return
    if pos not in game.state['blocked']:
        game.state['blocked'][pos] = True
        view.G.fill_rect(view.canvas, pos, view.grid_spec, color='black')


def on_mouse_out(pos):
    global is_mouse_down
    is_mouse_down = False


callbacks = {'s': game.search,
             'b': toggle_blockmode,
             'k': toggle_knightmoves,
             '+': upsize,
             '-': downsize,
             'r': game.reset or set_clicked(),       # loesche blocks, setze start=(0,0) und goal=(ncol,nrow)
             'p': view.draw_state or set_clicked(),  # zeige Such(p)roblem
             'u': lambda: (game.state['blocked'].pop(next(reversed(game.state['blocked'].keys())))
                           and view.draw_state()),
             'g': lambda: view.G.draw_grid(view.canvas, view.grid_spec),
             '1': lambda: game.set_searcher(game.S.search_df),
             '2': lambda: game.set_searcher(game.S.search_bf),
             '3': lambda: game.set_searcher(game.S.search_greedy),
             '4': lambda: game.set_searcher(game.S.search_smart),
             '5': lambda: game.set_searcher(game.S.search_bibf),
             '6': lambda: game.set_searcher(game.S.search_bi_smart),
             'mouse_down': on_mouse_down,
             'mouse_up': on_mouse_up,
             'mouse_move': on_mouse_move,
             'mouse_out': on_mouse_out,
             }

view.init(game)
C.init(view.canvas, callbacks, view.grid_spec)
C.show(debug=True)

In [ ]:
# Heuristik ueberschreiben
# game.H = lambda pos, goal: 7 if pos == (2, 0) else 0